# Olivia Rodrigo Production Evolution — A Data Engineering Analysis

This notebook builds a reproducible pipeline that tracks how Olivia Rodrigo's production quality and audio characteristics evolved across her three studio albums — **SOUR** (2021), **GUTS** (2023), and **you seem pretty sad for a girl so in love** (2026) — all produced by Dan Nigro, and tests whether that evolution correlates with audience engagement (Spotify popularity, chart performance).

**Structure:**
1. Data Ingestion
2. Data Cleaning / ETL
3. Exploratory Analysis
4. Engagement Correlation
5. (Code quality is applied throughout, not a separate section)

**A note on data provenance**, since it shapes everything below: Spotify deprecated public access to `/audio-features` and `/audio-analysis` for every app created after Nov 27, 2024 — there is no credential that unlocks it. Combined with the fact that no dataset anywhere can contain audio features for a June 2026 album, this notebook treats audio features as **manually/partially sourced** rather than pretending a clean API exists. Track identity data (names, order, duration) is sourced from Wikipedia rather than the Spotify API, and live Spotify data is used only transiently at runtime for popularity scores — per this project's Spotify Developer Terms compliance decision, raw Spotify API responses are never written to disk; only the final point-in-time popularity number (dated and attributed) flows into the processed dataset. See the Data Ingestion section for the full source-by-source breakdown.

## Setup

In [1]:
import os
import re
import time
import base64
from pathlib import Path
from datetime import datetime, timezone

import truststore
# Uses the OS certificate trust store instead of the bundled `certifi` list. Needed on
# machines where antivirus/network software (e.g. Avast Web Shield) does TLS inspection —
# it re-signs HTTPS traffic with a locally-generated root CA that Windows trusts but
# certifi doesn't. This makes Python trust what the OS already trusts; it does NOT
# disable certificate verification.
truststore.inject_into_ssl()

import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project paths, resolved relative to this notebook so it works regardless of the
# directory the Jupyter server was launched from.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print(f"Project root:     {PROJECT_ROOT}")
print(f"Raw data dir:      {RAW_DIR}")
print(f"Processed data dir: {PROCESSED_DIR}")

Project root:     D:\OR analysis
Raw data dir:      D:\OR analysis\data\raw
Processed data dir: D:\OR analysis\data\processed


## 1. Data Ingestion

Four sources feed this pipeline, each with a distinct role and distinct limitations:

| Source | Provides | Persisted to `/data/raw`? |
|---|---|---|
| Wikipedia (hardcoded, cited) | Track names, order, duration, bonus-track flags | Yes |
| Spotify Web API (Client Credentials) | Popularity score (live only) | No — Developer Terms prohibit indefinite storage of Spotify content; used in-memory and only the final dated snapshot reaches `/data/processed` |
| GetSongBPM API | Tempo, key, time signature | Yes |
| Manual entry | Energy, loudness, valence, acousticness, danceability, instrumentalness, speechiness | Yes (template, blank until filled) |

Each source is wrapped behind its own small class/function so the pipeline is swappable — if a better free audio-feature API appears later, only the relevant block needs to change, not the downstream cleaning/analysis code.

### 1.1 Track identity (Wikipedia)

Spotify's own catalog endpoints (`Get Album`, `Get Album Tracks`) could technically supply this, but since track names/order/duration would then count as persisted Spotify Content under the Developer Terms, this project sources them from Wikipedia instead — publicly citable, and avoids the compliance question entirely for the one thing that has to live in a committed CSV either way.

In [2]:
def mmss_to_seconds(mmss: str) -> int:
    """Convert an 'M:SS' duration string to total seconds."""
    minutes, seconds = mmss.split(":")
    return int(minutes) * 60 + int(seconds)


# Canonical track listings, sourced from Wikipedia (accessed 2026-07-17):
#   https://en.wikipedia.org/wiki/Sour_(album)
#   https://en.wikipedia.org/wiki/Guts_(album)
#   https://en.wikipedia.org/wiki/You_Seem_Pretty_Sad_for_a_Girl_So_in_Love
#
# Deluxe/bonus-track rule (documented per project spec): bonus and deluxe tracks are
# INCLUDED, not dropped, but flagged via `is_bonus=True`. Downstream analysis can
# then choose standard-only or full-including-deluxe per chart without losing data.
ALBUMS_RAW = {
    "SOUR": {
        "release_date": "2021-05-21",
        "era": 1,
        "tracks": [
            (1, "Brutal", "2:23", False),
            (2, "Traitor", "3:49", False),
            (3, "Drivers License", "4:02", False),
            (4, "1 Step Forward, 3 Steps Back", "2:43", False),
            (5, "Deja Vu", "3:35", False),
            (6, "Good 4 U", "2:58", False),
            (7, "Enough for You", "3:22", False),
            (8, "Happier", "2:55", False),
            (9, "Jealousy, Jealousy", "2:53", False),
            (10, "Favorite Crime", "2:32", False),
            (11, "Hope Ur Ok", "3:29", False),
        ],
    },
    "GUTS": {
        "release_date": "2023-09-08",
        "era": 2,
        "tracks": [
            (1, "All-American Bitch", "2:45", False),
            (2, "Bad Idea Right?", "3:04", False),
            (3, "Vampire", "3:39", False),
            (4, "Lacy", "2:57", False),
            (5, "Ballad of a Homeschooled Girl", "3:23", False),
            (6, "Making the Bed", "3:18", False),
            (7, "Logical", "3:51", False),
            (8, "Get Him Back!", "3:31", False),
            (9, "Love Is Embarrassing", "2:34", False),
            (10, "The Grudge", "3:09", False),
            (11, "Pretty Isn't Pretty", "3:19", False),
            (12, "Teenage Dream", "3:42", False),
            # GUTS (spilled) deluxe edition, released 2024-03-22
            (13, "Obsessed", "2:50", True),
            (14, "Girl I've Always Been", "2:01", True),
            (15, "Scared of My Guitar", "4:23", True),
            (16, "Stranger", "3:12", True),
            (17, "So American", "2:49", True),
        ],
    },
    "you seem pretty sad for a girl so in love": {
        "release_date": "2026-06-12",
        "era": 3,
        "tracks": [
            (1, "Drop Dead", "3:44", False),
            (2, "Stupid Song", "3:29", False),
            (3, "Honeybee", "3:43", False),
            (4, "Maggots for Brains", "4:00", False),
            (5, "U + Me = <3", "4:07", False),
            (6, "My Way", "3:00", False),
            (7, "Purple", "4:00", False),
            (8, "The Cure", "4:57", False),
            (9, "Begged", "3:37", False),
            (10, "What's Wrong with Me (feat. Robert Smith)", "3:44", False),
            (11, "Less", "3:13", False),
            (12, "Expectations", "3:41", False),
            (13, "Cigarette Smoke", "5:40", False),
            # Bonus track added to the digital edition 2026-06-14
            (14, "Never Do", "3:43", True),
        ],
    },
}


def build_track_skeleton(albums_raw: dict) -> pd.DataFrame:
    """Flatten ALBUMS_RAW into one row per track — the spine every other source joins onto."""
    rows = []
    for album, meta in albums_raw.items():
        for track_number, title, duration, is_bonus in meta["tracks"]:
            rows.append({
                "album": album,
                "era": meta["era"],
                "release_date": meta["release_date"],
                "track_number": track_number,
                "title": title,
                "duration_sec": mmss_to_seconds(duration),
                "is_bonus": is_bonus,
            })
    return pd.DataFrame(rows)


track_skeleton = build_track_skeleton(ALBUMS_RAW)
print(f"Loaded {len(track_skeleton)} tracks across {track_skeleton['album'].nunique()} albums")
track_skeleton.head()

Loaded 42 tracks across 3 albums


,album,era,release_date,track_number,title,duration_sec,is_bonus
0,SOUR,1,2021-05-21,1,Brutal,143,False
1,SOUR,1,2021-05-21,2,Traitor,229,False
2,SOUR,1,2021-05-21,3,Drivers License,242,False
3,SOUR,1,2021-05-21,4,"1 Step Forward, 3 Steps Back",163,False
4,SOUR,1,2021-05-21,5,Deja Vu,215,False


### 1.2 Spotify Web API — popularity (live, not persisted)

Uses the **Client Credentials flow** (app-only, no user login) since everything pulled here — search, track popularity — is public catalog data, not user-specific data. This deliberately does *not* request any OAuth scopes.

This same client is also used to demonstrate, live, that `/audio-features` is dead: rather than asserting the Nov 2024 deprecation as a fact and skipping the call, it actually makes the request and reports whatever Spotify returns. Every request goes through a shared retry helper that respects `Retry-After` on HTTP 429 with exponential backoff, per Spotify's rate-limit guidance.

In [3]:
class SpotifyAuthError(Exception):
    """Raised when Spotify credentials are missing or rejected."""


class SpotifyClient:
    """
    Thin wrapper around the Spotify Web API's Client Credentials flow.

    Used ONLY for public catalog data (track search, popularity) — no user-specific
    scopes are requested, per Spotify's own guidance to reserve Client Credentials
    for non-user data.

    Per this project's Developer Terms compliance decision (see markdown above),
    this client is a live, in-memory lookup only. Nothing it returns is written to
    disk as a raw dump here; only a final popularity snapshot (numeric score + fetch
    date) is allowed to flow into the processed dataset later in the pipeline.
    """

    TOKEN_URL = "https://accounts.spotify.com/api/token"
    API_BASE = "https://api.spotify.com/v1"

    def __init__(self, client_id: str = None, client_secret: str = None):
        self.client_id = client_id or os.getenv("SPOTIFY_CLIENT_ID")
        self.client_secret = client_secret or os.getenv("SPOTIFY_CLIENT_SECRET")
        self._token = None
        self._token_expires_at = 0.0

    @property
    def is_configured(self) -> bool:
        return bool(self.client_id and self.client_secret)

    def _get_access_token(self) -> str:
        """Fetch (and cache) an app-only access token, refreshing shortly before it expires."""
        if not self.is_configured:
            raise SpotifyAuthError(
                "SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET not set in .env."
            )
        if self._token and time.time() < self._token_expires_at - 30:
            return self._token

        auth_bytes = f"{self.client_id}:{self.client_secret}".encode("utf-8")
        auth_header = base64.b64encode(auth_bytes).decode("utf-8")
        response = requests.post(
            self.TOKEN_URL,
            headers={
                "Authorization": f"Basic {auth_header}",
                "Content-Type": "application/x-www-form-urlencoded",
            },
            data={"grant_type": "client_credentials"},
            timeout=10,
        )
        if response.status_code != 200:
            raise SpotifyAuthError(
                f"Spotify token request failed ({response.status_code}): {response.text}"
            )
        payload = response.json()
        self._token = payload["access_token"]
        self._token_expires_at = time.time() + payload.get("expires_in", 3600)
        return self._token

    def _request(self, method: str, path: str, **kwargs):
        """
        Authenticated request with exponential backoff on HTTP 429, honoring the
        Retry-After header per Spotify's rate-limit guidance. Returns None (rather
        than raising) if retries are exhausted, so a single flaky call can't crash
        the whole ingestion run.
        """
        token = self._get_access_token()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {token}"

        backoff = 1.0
        for attempt in range(5):
            response = requests.request(
                method, f"{self.API_BASE}{path}", headers=headers, timeout=10, **kwargs
            )
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", backoff))
                print(f"    Rate limited by Spotify — waiting {retry_after}s (attempt {attempt + 1}/5)")
                time.sleep(retry_after)
                backoff *= 2
                continue
            return response
        print("    Gave up after repeated 429s from Spotify.")
        return None

    def search_track(self, title: str, artist: str = "Olivia Rodrigo"):
        """Resolve a track title to a Spotify track ID via the Search endpoint."""
        try:
            response = self._request(
                "GET", "/search",
                params={"q": f"track:{title} artist:{artist}", "type": "track", "limit": 1},
            )
        except SpotifyAuthError as exc:
            print(f"    Spotify auth unavailable: {exc}")
            return None
        if response is None:
            return None
        if response.status_code != 200:
            print(f"    Spotify search failed for '{title}' ({response.status_code}): {response.text[:200]}")
            return None
        items = response.json().get("tracks", {}).get("items", [])
        return items[0]["id"] if items else None

    def get_popularity_batch(self, track_ids: list) -> dict:
        """
        Fetch popularity for up to 50 track IDs at once via Get Several Tracks.
        `popularity` is a live field here — unlike the deprecated copy nested inside
        Get Album's track list.
        """
        popularity = {}
        clean_ids = [t for t in track_ids if t]
        for i in range(0, len(clean_ids), 50):
            batch = clean_ids[i:i + 50]
            try:
                response = self._request("GET", "/tracks", params={"ids": ",".join(batch)})
            except SpotifyAuthError as exc:
                print(f"    Spotify auth unavailable: {exc}")
                return popularity
            if response is None or response.status_code != 200:
                status = response.status_code if response is not None else "no response"
                print(f"    Spotify Get Several Tracks failed ({status})")
                continue
            for track in response.json().get("tracks", []):
                if track:
                    popularity[track["id"]] = track["popularity"]
        return popularity

    def attempt_audio_features(self, sample_track_id: str):
        """
        Demonstrates, live, rather than assumes, the Nov 2024 deprecation: calls
        /audio-features for one real track ID and reports exactly what comes back.
        Expected result for any app created after Nov 27 2024: HTTP 403.
        """
        try:
            response = self._request("GET", f"/audio-features/{sample_track_id}")
        except SpotifyAuthError as exc:
            print(f"    Spotify auth unavailable: {exc}")
            return None
        if response is None:
            return None
        if response.status_code == 200:
            print("    /audio-features succeeded — this app has pre-Nov-2024 extended quota.")
            return response.json()
        print(
            f"    /audio-features returned {response.status_code} as expected for a "
            f"post-deprecation app: {response.text[:200]}"
        )
        return None

In [4]:
spotify = SpotifyClient()
spotify_popularity_snapshot = pd.DataFrame(columns=["title", "album", "spotify_popularity", "fetched_on"])

if not spotify.is_configured:
    print(
        "No Spotify credentials found in .env — skipping live Spotify calls entirely.\n"
        "Popularity scores will be left blank in the processed dataset. To fill them in,\n"
        "add SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET to .env and re-run this cell."
    )
else:
    try:
        spotify._get_access_token()
        print("Spotify authentication succeeded.\n")
    except SpotifyAuthError as exc:
        print(f"Spotify authentication failed: {exc}")
        spotify = SpotifyClient(client_id=None, client_secret=None)  # force is_configured False below

if spotify.is_configured:
    print("Resolving Spotify track IDs (search, in-memory only)...")
    spotify_track_ids = {}
    for _, row in track_skeleton.iterrows():
        spotify_track_ids[(row["album"], row["title"])] = spotify.search_track(row["title"])
        time.sleep(0.05)  # light client-side pacing on top of the 429 backoff in _request

    resolved_ids = [tid for tid in spotify_track_ids.values() if tid]
    print(f"Resolved {len(resolved_ids)}/{len(track_skeleton)} tracks to Spotify IDs.")

    print("\nFetching popularity scores (live, not cached to disk)...")
    popularity_by_id = spotify.get_popularity_batch(resolved_ids)

    fetched_on = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    rows = [
        {"title": title, "album": album, "spotify_popularity": popularity_by_id[track_id], "fetched_on": fetched_on}
        for (album, title), track_id in spotify_track_ids.items()
        if track_id and track_id in popularity_by_id
    ]
    spotify_popularity_snapshot = pd.DataFrame(rows)
    print(f"Got popularity for {len(spotify_popularity_snapshot)} tracks.")

    if resolved_ids:
        print("\nConfirming /audio-features status on a live sample track...")
        spotify.attempt_audio_features(resolved_ids[0])

spotify_popularity_snapshot.head()

No Spotify credentials found in .env — skipping live Spotify calls entirely.
Popularity scores will be left blank in the processed dataset. To fill them in,
add SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET to .env and re-run this cell.


,title,album,spotify_popularity,fetched_on


### 1.3 GetSongBPM — tempo, key, time signature (persisted)

Not Spotify content, so no storage restriction applies — this is written straight to `/data/raw`.

**Live result:** GetSongBPM's endpoint sits behind Cloudflare bot protection and returns an HTML JS-challenge page (HTTP 403) to plain server-side requests, independent of API key validity — confirmed by testing directly against the endpoint outside this notebook. This project doesn't attempt to work around Cloudflare's challenge (that would mean impersonating a browser to bypass bot protection), so `GetSongBPMSource` detects this condition once and fails fast rather than retrying it per track. In practice, tempo/key end up manual too, alongside the features no free source ever covered.

In [5]:
class GetSongBPMSource:
    """
    Wrapper around the free GetSongBPM API (https://getsongbpm.com/api).
    Provides tempo, key, and time signature — NOT the full Spotify-style feature set
    (no energy, loudness, valence, instrumentalness, or speechiness; those stay manual).

    GetSongBPM's endpoint sits behind Cloudflare bot protection, which can return an
    HTML JS-challenge page (HTTP 403) to any plain server-side request regardless of
    API key validity. This is detected explicitly and treated as a fail-fast condition
    for the whole source, rather than retried per track — attempting to work around
    Cloudflare's challenge would mean impersonating a browser to bypass bot protection,
    which this project won't do. When blocked, remaining audio features stay manual.
    """

    SEARCH_URL = "https://api.getsongbpm.com/search/"

    def __init__(self, api_key: str = None):
        self.api_key = api_key or os.getenv("GETSONGBPM_API_KEY")
        self._blocked_reason = None

    @property
    def is_configured(self) -> bool:
        return bool(self.api_key)

    @property
    def is_blocked(self) -> bool:
        return self._blocked_reason is not None

    def lookup(self, title: str, artist: str = "Olivia Rodrigo", _retry: bool = True):
        if not self.is_configured or self.is_blocked:
            return None
        try:
            response = requests.get(
                self.SEARCH_URL,
                params={
                    "api_key": self.api_key,
                    "type": "song",
                    "lookup": f"song:{title} artist:{artist}",
                },
                timeout=10,
            )
        except requests.RequestException as exc:
            print(f"    GetSongBPM request error for '{title}': {exc}")
            return None

        if response.status_code == 429 and _retry:
            retry_after = int(response.headers.get("Retry-After", 5))
            print(f"    GetSongBPM rate limited — waiting {retry_after}s")
            time.sleep(retry_after)
            return self.lookup(title, artist, _retry=False)

        if response.status_code == 403 and "cloudflare" in response.text.lower():
            self._blocked_reason = (
                "GetSongBPM's endpoint returned a Cloudflare bot-challenge page (HTTP 403) "
                "instead of JSON — this happens for any plain server-side request, "
                "independent of API key validity. Treating this source as unavailable for "
                "the rest of this run; tempo/key fall through to manual entry."
            )
            print(f"    {self._blocked_reason}")
            return None

        if response.status_code != 200:
            print(f"    GetSongBPM lookup failed for '{title}' ({response.status_code})")
            return None

        try:
            results = response.json().get("search", [])
        except ValueError:
            print(f"    GetSongBPM returned a non-JSON payload for '{title}' — skipping")
            return None
        if not isinstance(results, list) or not results:
            return None

        match = results[0]
        return {
            "tempo": match.get("tempo"),
            "key": match.get("key_of"),
            "time_signature": match.get("time_sig"),
            "danceability": match.get("danceability"),
            "acousticness": match.get("acousticness"),
        }


getsongbpm = GetSongBPMSource()
getsongbpm_rows = []

if getsongbpm.is_configured:
    print("Looking up tempo/key via GetSongBPM (one request per second, free-tier pacing)...")
    for _, row in track_skeleton.iterrows():
        if getsongbpm.is_blocked:
            getsongbpm_rows.append({"album": row["album"], "title": row["title"]})
            continue
        result = getsongbpm.lookup(row["title"])
        getsongbpm_rows.append({"album": row["album"], "title": row["title"], **(result or {})})
        if not getsongbpm.is_blocked:
            time.sleep(1)
    getsongbpm_df = pd.DataFrame(getsongbpm_rows)
    matched = getsongbpm_df["tempo"].notna().sum() if "tempo" in getsongbpm_df.columns else 0
    print(f"Matched {matched}/{len(track_skeleton)} tracks on GetSongBPM.")
else:
    print("No GETSONGBPM_API_KEY set — skipping. Tempo/key will be left for manual entry.")
    getsongbpm_df = pd.DataFrame(
        columns=["album", "title", "tempo", "key", "time_signature", "danceability", "acousticness"]
    )

getsongbpm_df.head()

Looking up tempo/key via GetSongBPM (one request per second, free-tier pacing)...


    GetSongBPM's endpoint returned a Cloudflare bot-challenge page (HTTP 403) instead of JSON — this happens for any plain server-side request, independent of API key validity. Treating this source as unavailable for the rest of this run; tempo/key fall through to manual entry.
Matched 0/42 tracks on GetSongBPM.


,album,title
0,SOUR,Brutal
1,SOUR,Traitor
2,SOUR,Drivers License
3,SOUR,"1 Step Forward, 3 Steps Back"
4,SOUR,Deja Vu


### 1.4 Manual audio-feature template + raw CSV export

Merges the Wikipedia track spine with whatever GetSongBPM matched, adds blank columns for the features no free source covers, and writes one CSV per album to `/data/raw`. The `notes` column documents exactly how to fill the gaps so methodology stays consistent across all three albums (and reproducible for anyone reading the eventual article).

In [6]:
# Features no free source covers at all — always manual.
MANUAL_ONLY_COLUMNS = ["energy", "loudness", "valence", "instrumentalness", "speechiness"]
# Features GetSongBPM *can* cover but may miss per-track — manual is the fallback, not the default.
PARTIAL_COVERAGE_COLUMNS = ["tempo", "key", "time_signature", "danceability", "acousticness"]

raw_df = track_skeleton.merge(getsongbpm_df, on=["album", "title"], how="left")

for col in MANUAL_ONLY_COLUMNS + PARTIAL_COVERAGE_COLUMNS:
    if col not in raw_df.columns:
        raw_df[col] = np.nan

raw_df["notes"] = (
    "Fill blank audio-feature columns by looking up '<title> Olivia Rodrigo' on a tool "
    "like tunebat.com or chosic.com and recording the displayed values. Note which tool "
    "you used per row if it varies, for reproducibility."
)

print("Column completeness before manual entry:")
for col in MANUAL_ONLY_COLUMNS + PARTIAL_COVERAGE_COLUMNS:
    present = raw_df[col].notna().sum()
    print(f"  {col:16s} {present:2d}/{len(raw_df)} filled")

for album in raw_df["album"].unique():
    album_slug = re.sub(r"[^a-z0-9]+", "_", album.lower()).strip("_")
    out_path = RAW_DIR / f"{album_slug}_raw.csv"
    raw_df.loc[raw_df["album"] == album].to_csv(out_path, index=False)
    print(f"Saved {out_path}")

Column completeness before manual entry:
  energy            0/42 filled
  loudness          0/42 filled
  valence           0/42 filled
  instrumentalness  0/42 filled
  speechiness       0/42 filled
  tempo             0/42 filled
  key               0/42 filled
  time_signature    0/42 filled
  danceability      0/42 filled
  acousticness      0/42 filled
Saved D:\OR analysis\data\raw\sour_raw.csv
Saved D:\OR analysis\data\raw\guts_raw.csv
Saved D:\OR analysis\data\raw\you_seem_pretty_sad_for_a_girl_so_in_love_raw.csv


### 1.5 Chart performance (Wikipedia/Billboard, cited)

No free public API exposes Billboard chart history, so these are compiled by hand from Billboard.com coverage and Wikipedia (accessed 2026-07-17), the same standard a published article would cite to.

In [7]:
# Sourced from Billboard.com chart coverage / Wikipedia (accessed 2026-07-17):
#   "drivers license", "good 4 u" — Billboard Hot 100 #1 debuts
#   "deja vu" — Billboard Hot 100 #3 peak
#   "vampire" — Billboard Hot 100 #1 (two separate weeks at #1)
#   "bad idea right?" — Billboard Hot 100 #7 peak
#   "get him back!" — Billboard Hot 100 #11 peak
#   "Drop Dead", "The Cure", "Stupid Song" — per Wikipedia's album/single pages
CHART_PEAKS = pd.DataFrame([
    {"album": "SOUR", "single": "Drivers License", "single_order": 1, "billboard_hot100_peak": 1},
    {"album": "SOUR", "single": "Deja Vu", "single_order": 2, "billboard_hot100_peak": 3},
    {"album": "SOUR", "single": "Good 4 U", "single_order": 3, "billboard_hot100_peak": 1},
    {"album": "GUTS", "single": "Vampire", "single_order": 1, "billboard_hot100_peak": 1},
    {"album": "GUTS", "single": "Bad Idea Right?", "single_order": 2, "billboard_hot100_peak": 7},
    {"album": "GUTS", "single": "Get Him Back!", "single_order": 3, "billboard_hot100_peak": 11},
    {"album": "you seem pretty sad for a girl so in love", "single": "Drop Dead", "single_order": 1, "billboard_hot100_peak": 1},
    {"album": "you seem pretty sad for a girl so in love", "single": "The Cure", "single_order": 2, "billboard_hot100_peak": 5},
    {"album": "you seem pretty sad for a girl so in love", "single": "Stupid Song", "single_order": 3, "billboard_hot100_peak": 3},
])
CHART_PEAKS.to_csv(RAW_DIR / "chart_peaks.csv", index=False)
print(f"Saved {RAW_DIR / 'chart_peaks.csv'}")
CHART_PEAKS

Saved D:\OR analysis\data\raw\chart_peaks.csv


,album,single,single_order,billboard_hot100_peak
0,SOUR,Drivers License,1,1
1,SOUR,Deja Vu,2,3
2,SOUR,Good 4 U,3,1
3,GUTS,Vampire,1,1
4,GUTS,Bad Idea Right?,2,7
5,GUTS,Get Him Back!,3,11
6,you seem pretty sad for a girl so in love,Drop Dead,1,1
7,you seem pretty sad for a girl so in love,The Cure,2,5
8,you seem pretty sad for a girl so in love,Stupid Song,3,3


## 2. Data Cleaning / ETL

This section deliberately re-reads the CSVs just written to `/data/raw`, rather than reusing the in-memory `raw_df` from Section 1 — that's what makes it a real, independent ETL stage: it works whether or not Section 1 ran in this session, including after you've hand-filled the manual audio-feature columns.

**Deluxe/bonus/remix rule** (documented once here, applied consistently): bonus and deluxe tracks are **kept, not dropped**, and flagged via `is_bonus`. Downstream charts and stats default to standard-edition tracks only (`is_bonus == False`) for cross-album comparability — GUTS's 5 deluxe tracks were recorded months after the standard edition and would otherwise skew "GUTS-era" audio characteristics — but nothing is discarded, so any cell can opt into `include_bonus=True` and get the full catalog. None of the three albums has an official remix on the studio tracklist, so no remix-specific rule is needed.

In [8]:
# Expected value ranges, used only to flag likely data-entry mistakes when the manual
# columns get filled in — never to reject or silently alter data.
FEATURE_BOUNDS = {
    "energy": (0.0, 1.0),
    "valence": (0.0, 1.0),
    "danceability": (0.0, 1.0),
    "acousticness": (0.0, 1.0),
    "instrumentalness": (0.0, 1.0),
    "speechiness": (0.0, 1.0),
    "loudness": (-60.0, 0.0),
    "tempo": (40.0, 220.0),
    "time_signature": (3, 7),
    "spotify_popularity": (0, 100),
}


def load_raw_tracks(raw_dir: Path) -> pd.DataFrame:
    """Read every *_raw.csv in data/raw and concatenate — an independent ETL entry point."""
    csv_paths = sorted(raw_dir.glob("*_raw.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No *_raw.csv files found in {raw_dir} — run Section 1 first.")
    combined = pd.concat([pd.read_csv(path) for path in csv_paths], ignore_index=True)
    print(f"Loaded {len(combined)} tracks from {len(csv_paths)} raw file(s): {[p.name for p in csv_paths]}")
    return combined


def coerce_numeric(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Force the listed columns to numeric dtype. Anything unparseable (typos, stray
    text from manual entry) becomes NaN instead of silently leaving a mixed-type
    column — and gets reported, so a bad manual entry is visible, not swallowed.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        coerced = pd.to_numeric(df[col], errors="coerce")
        bad_mask = df[col].notna() & coerced.isna()
        if bad_mask.any():
            print(f"  '{col}': could not parse {df.loc[bad_mask, col].tolist()} as numbers — set to NaN")
        df[col] = coerced
    return df


def validate_tracks(df: pd.DataFrame) -> None:
    """Print missing-value, out-of-range, duplicate, and sequencing checks."""
    print(f"Rows: {len(df)}  |  Columns: {len(df.columns)}")

    print("\n--- Missing values ---")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print("  None.")
    else:
        for col, count in missing.items():
            print(f"  {col:20s} {count:2d}/{len(df)} missing ({count / len(df):.0%})")

    print("\n--- Out-of-range values (vs. expected bounds, flags only) ---")
    any_outliers = False
    for col, (low, high) in FEATURE_BOUNDS.items():
        if col not in df.columns:
            continue
        bad = df[df[col].notna() & ~df[col].between(low, high)]
        for _, row in bad.iterrows():
            any_outliers = True
            print(f"  {row['album']} — '{row['title']}': {col}={row[col]} outside expected [{low}, {high}]")
    if not any_outliers:
        print("  None.")

    print("\n--- Duplicate (album, title) pairs ---")
    dupes = df[df.duplicated(subset=["album", "title"], keep=False)]
    print("  None." if dupes.empty else dupes[["album", "title"]].to_string(index=False))

    print("\n--- Track-number sequencing per album (standard edition only) ---")
    for album, group in df[~df["is_bonus"]].groupby("album"):
        expected = list(range(1, len(group) + 1))
        actual = sorted(group["track_number"].tolist())
        status = "OK" if actual == expected else f"MISMATCH — got {actual}, expected {expected}"
        print(f"  {album}: {status}")


raw_tracks = load_raw_tracks(RAW_DIR)
raw_tracks = coerce_numeric(raw_tracks, list(FEATURE_BOUNDS.keys()))
print()
validate_tracks(raw_tracks)

Loaded 42 tracks from 3 raw file(s): ['guts_raw.csv', 'sour_raw.csv', 'you_seem_pretty_sad_for_a_girl_so_in_love_raw.csv']

Rows: 42  |  Columns: 18

--- Missing values ---
  energy               42/42 missing (100%)
  loudness             42/42 missing (100%)
  valence              42/42 missing (100%)
  instrumentalness     42/42 missing (100%)
  speechiness          42/42 missing (100%)
  tempo                42/42 missing (100%)
  key                  42/42 missing (100%)
  time_signature       42/42 missing (100%)
  danceability         42/42 missing (100%)
  acousticness         42/42 missing (100%)

--- Out-of-range values (vs. expected bounds, flags only) ---
  None.

--- Duplicate (album, title) pairs ---
  None.

--- Track-number sequencing per album (standard edition only) ---
  GUTS: OK
  SOUR: OK
  you seem pretty sad for a girl so in love: OK


### 2.1 Merge in popularity and save `tracks_clean.csv`

Popularity is the one field that can't be reloaded from disk independently of Section 1 — by design, per the Spotify Developer Terms decision in Section 1.2, it only ever exists in-memory for the current run. If Section 1 hasn't executed in this kernel session, it's treated as unavailable here rather than assumed to be zero or dropped silently.

In [9]:
try:
    _popularity = spotify_popularity_snapshot
except NameError:
    print(
        "Section 1 hasn't run in this kernel session — spotify_popularity_snapshot is "
        "unavailable, so popularity will be blank below. Run Section 1 first to populate it."
    )
    _popularity = pd.DataFrame(columns=["album", "title", "spotify_popularity", "fetched_on"])

tracks_clean = raw_tracks.merge(
    _popularity[["album", "title", "spotify_popularity", "fetched_on"]], on=["album", "title"], how="left"
)
tracks_clean = coerce_numeric(tracks_clean, ["spotify_popularity"])

# Identity -> audio features -> engagement -> metadata, in that reading order.
COLUMN_ORDER = [
    "album", "era", "release_date", "track_number", "title", "is_bonus", "duration_sec",
    "energy", "loudness", "valence", "danceability", "acousticness", "instrumentalness",
    "speechiness", "tempo", "key", "time_signature",
    "spotify_popularity", "fetched_on", "notes",
]
tracks_clean = tracks_clean[[c for c in COLUMN_ORDER if c in tracks_clean.columns]]
tracks_clean = tracks_clean.sort_values(["era", "track_number"]).reset_index(drop=True)

print("--- Column dtypes ---")
print(tracks_clean.dtypes.to_string())

have_pop = tracks_clean["spotify_popularity"].notna().sum()
print(f"\n{have_pop}/{len(tracks_clean)} tracks have a Spotify popularity score.")
if have_pop == 0:
    print("(Add SPOTIFY_CLIENT_ID/SECRET to .env and re-run Section 1.2 to populate this.)")

tracks_clean.to_csv(PROCESSED_DIR / "tracks_clean.csv", index=False)
print(f"\nSaved {PROCESSED_DIR / 'tracks_clean.csv'} — {len(tracks_clean)} rows, {len(tracks_clean.columns)} columns")
tracks_clean.head(10)

--- Column dtypes ---
album                  object
era                     int64
release_date           object
track_number            int64
title                  object
is_bonus                 bool
duration_sec            int64
energy                float64
loudness              float64
valence               float64
danceability          float64
acousticness          float64
instrumentalness      float64
speechiness           float64
tempo                 float64
key                   float64
time_signature        float64
spotify_popularity    float64
fetched_on             object
notes                  object

0/42 tracks have a Spotify popularity score.
(Add SPOTIFY_CLIENT_ID/SECRET to .env and re-run Section 1.2 to populate this.)

Saved D:\OR analysis\data\processed\tracks_clean.csv — 42 rows, 20 columns


,album,era,release_date,track_number,title,is_bonus,duration_sec,energy,loudness,valence,danceability,acousticness,instrumentalness,speechiness,tempo,key,time_signature,spotify_popularity,fetched_on,notes
0,SOUR,1,2021-05-21,1,Brutal,False,143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
1,SOUR,1,2021-05-21,2,Traitor,False,229,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
2,SOUR,1,2021-05-21,3,Drivers License,False,242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
3,SOUR,1,2021-05-21,4,"1 Step Forward, 3 Steps Back",False,163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
4,SOUR,1,2021-05-21,5,Deja Vu,False,215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
5,SOUR,1,2021-05-21,6,Good 4 U,False,178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
6,SOUR,1,2021-05-21,7,Enough for You,False,202,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
7,SOUR,1,2021-05-21,8,Happier,False,175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
8,SOUR,1,2021-05-21,9,"Jealousy, Jealousy",False,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
9,SOUR,1,2021-05-21,10,Favorite Crime,False,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fill blank audio-feature columns by looking up...
